# バイオ技術 1-3：シングルセルRNA-seq解析の基礎

このNotebookでは、ヒト末梢血単核細胞（PBMC）のsingle-cell RNA-seqデータを使って、

**1細胞ごとの遺伝子発現データから、異なる細胞集団を見つける流れ**

を体験します。

1-2では、bulk RNA-seqを使って**サンプル間の違い**を比較しました。  
ここでは、single-cell RNA-seqを使って**細胞1個1個の違い**を調べます。

---

## 今日のゴール

このNotebookが終わるころには、次のことを説明できるようになることを目指します。

1. bulk RNA-seqとsingle-cell RNA-seqの違い
2. QC（品質管理）がなぜ必要か
3. UMAPの1つ1つの点が何を表すか
4. クラスタリングとは何をしているのか
5. マーカー遺伝子から細胞型を推定する考え方

---

## このNotebookの進め方

途中に4つのミニ演習があります。

- QCの閾値を変えて、残る細胞数を比較する
- UMAPを見て、クラスタ数を予想する
- Leiden clusteringのresolutionを変える
- 自分でマーカー遺伝子を選び、UMAP上の発現を見る

**実行する → 結果を見る → 条件を変える → 自分で考える**

を意識して進めてください。


---
## 0. bulk RNA-seqとsingle-cell RNA-seqの違い

### bulk RNA-seq

多数の細胞から得られたRNAをまとめて測定します。

例えば、組織の中にT細胞、B細胞、単球が混ざっていた場合、得られる発現量はそれらをまとめた平均的な情報になります。

### single-cell RNA-seq

1細胞ごとに遺伝子発現を測定します。

そのため、

- どのような細胞集団が存在するか
- 同じように見える細胞の中に異なる集団があるか
- それぞれの細胞集団で、どの遺伝子が特徴的に発現しているか

を調べることができます。

今回の研究の問いは、

> **PBMCの中には、遺伝子発現パターンが異なるどのような細胞集団が存在するか？**

です。


---
## 1. Scanpyを準備する

single-cell RNA-seq解析には、Pythonの`Scanpy`というライブラリを使います。

最初のセルでは、必要なライブラリをインストールします。

このNotebookでは、Google Colabの標準環境との互換性を保つために、

- `pandas==2.2.2`
- `scanpy[leiden]==1.11.5`

を指定してインストールします。

`[leiden]`を指定すると、Leiden clusteringに必要な追加パッケージもまとめてインストールされます。

> インストールには少し時間がかかることがあります。


In [ ]:
%pip -q install "pandas==2.2.2" "scanpy[leiden]==1.11.5"


必要なライブラリを読み込みます。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

# 表示設定
sc.settings.verbosity = 2
sc.set_figure_params(figsize=(6, 5), dpi=100)

print("pandas version:", pd.__version__)
print("Scanpy version:", sc.__version__)


---
## 2. PBMC3Kデータを読み込む

今回は、PBMC（Peripheral Blood Mononuclear Cells：末梢血単核細胞）の公開single-cell RNA-seqデータを使います。

PBMCには、T細胞、B細胞、NK細胞、単球など複数の免疫細胞が含まれています。

下のセルでデータを読み込みます。


In [ ]:
adata = sc.datasets.pbmc3k()
adata


### AnnDataとは？

Scanpyでは、single-cell RNA-seqデータを`AnnData`という形式で扱います。

Excelの表に少し似ていますが、発現量だけでなく、細胞や遺伝子の情報もまとめて保存できます。

| 要素 | 内容 |
|---|---|
| `adata.X` | 発現量の行列 |
| `adata.obs` | 細胞ごとの情報 |
| `adata.var` | 遺伝子ごとの情報 |
| `adata.obsm` | PCAやUMAPなどの座標 |

基本的には、

- **行：細胞**
- **列：遺伝子**

です。

### 確認してみよう

下のセルで、細胞数と遺伝子数を確認します。


In [ ]:
n_cells, n_genes = adata.shape

print("細胞数:", n_cells)
print("遺伝子数:", n_genes)


### ミニ確認

表示された結果を記録してください。

- 細胞数：
- 遺伝子数：

1-2のbulk RNA-seqでは、少数のサンプルを比較しました。  
今回は、数千個の細胞を1つずつ比較します。


---
## 3. QC（品質管理）のための情報を見る

single-cell RNA-seqでは、すべての細胞をそのまま解析に使うわけではありません。

例えば、

- 検出された遺伝子数が極端に少ない細胞
- ミトコンドリア遺伝子の割合が非常に高い細胞
- 異常に多くの遺伝子が検出された細胞

などは、解析結果に影響する可能性があります。

まず、各細胞のQC指標を計算します。


In [ ]:
# ミトコンドリア遺伝子を識別する
adata.var["mt"] = adata.var_names.str.startswith("MT-")

# QC指標を計算する
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt"],
    percent_top=None,
    log1p=False,
    inplace=True
)

# 各細胞のQC情報を確認する
adata.obs[["n_genes_by_counts", "total_counts", "pct_counts_mt"]].head()


QC指標の分布を図で確認します。

- `n_genes_by_counts`：その細胞で検出された遺伝子数
- `total_counts`：その細胞で得られた総カウント数
- `pct_counts_mt`：ミトコンドリア遺伝子の割合


In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True
)


### 見るポイント

- 多くの細胞が集まっている範囲はどこですか？
- 極端な値を持つ細胞はありますか？
- どこを閾値にするかで、残る細胞数は変わりそうですか？

QCでは「絶対にこの閾値が正しい」というより、データの分布を確認して基準を決めることが重要です。


### ミニ演習1：ミトコンドリア割合の閾値を変えてみる

まだ実際のデータは削除しません。

下の`TEST_MT_THRESHOLD`を書き換えて、閾値によって何個の細胞が残るかを比べてください。

例えば、

- 3%
- 5%
- 10%

を試してみましょう。

**予想してから実行してください。**

> 閾値を厳しくすると、残る細胞数は増えるでしょうか、減るでしょうか？


In [ ]:
# ↓↓↓ ここを書き換えてください ↓↓↓
TEST_MT_THRESHOLD = 5.0

n_before = adata.n_obs
n_after = int((adata.obs["pct_counts_mt"] < TEST_MT_THRESHOLD).sum())

print("QC前の細胞数:", n_before)
print("設定した閾値:", TEST_MT_THRESHOLD, "%")
print("条件を満たす細胞数:", n_after)
print("除外される細胞数:", n_before - n_after)


#### 記録してみよう

| mt閾値 | 残る細胞数 |
|---|---|
| 3% | |
| 5% | |
| 10% | |

- 閾値を厳しくすると、残る細胞数はどうなりましたか？
- QCの基準によって解析対象が変わることを確認できましたか？


---
## 4. 実際にフィルタリングする

この演習では、次の条件を使います。

- 200遺伝子未満しか検出されない細胞を除く
- 3細胞未満でしか検出されない遺伝子を除く
- 2,500遺伝子以上が検出された細胞を除く
- ミトコンドリア遺伝子割合が5%以上の細胞を除く

まず、フィルタリング前の細胞数を保存します。


In [ ]:
cells_before_qc = adata.n_obs
genes_before_qc = adata.n_vars

sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

adata = adata[
    (adata.obs["n_genes_by_counts"] < 2500)
    & (adata.obs["pct_counts_mt"] < 5),
    :
].copy()

print("QC前の細胞数:", cells_before_qc)
print("QC後の細胞数:", adata.n_obs)
print("除外された細胞数:", cells_before_qc - adata.n_obs)

print()
print("QC前の遺伝子数:", genes_before_qc)
print("QC後の遺伝子数:", adata.n_vars)


### 考えてみよう

QCを行うと、細胞数だけでなく遺伝子数も変わりました。

なぜでしょうか？

ヒント：  
ごく少数の細胞でしか検出されない遺伝子も除外しています。


---
## 5. 正規化とlog変換

細胞ごとに得られる総カウント数は異なります。

そのまま比較すると、

> 本当に発現が高い細胞

と

> 単に多く読まれただけの細胞

を区別しにくくなります。

そこで、各細胞の総カウントをそろえる正規化を行います。


In [ ]:
# 各細胞の総カウントを10,000にそろえる
sc.pp.normalize_total(adata, target_sum=1e4)

# log(1 + x)変換
sc.pp.log1p(adata)


ここで、後のマーカー遺伝子解析に使えるように、正規化・log変換後の全遺伝子データを保存しておきます。


In [ ]:
adata.raw = adata


---
## 6. 高変動遺伝子を選ぶ

すべての遺伝子が、細胞の違いを見分けるために同じくらい重要とは限りません。

そこで、

> **細胞間で発現量の違いが大きい遺伝子**

を選びます。これを高変動遺伝子（Highly Variable Genes: HVGs）と呼びます。


In [ ]:
sc.pp.highly_variable_genes(
    adata,
    min_mean=0.0125,
    max_mean=3,
    min_disp=0.5
)

print("全遺伝子数:", adata.n_vars)
print("高変動遺伝子数:", int(adata.var["highly_variable"].sum()))

sc.pl.highly_variable_genes(adata)


高変動遺伝子だけを残し、PCAやUMAPの計算に使います。


In [ ]:
adata = adata[:, adata.var["highly_variable"]].copy()

# 各遺伝子を平均0、分散1程度にそろえる
sc.pp.scale(adata, max_value=10)

print("解析に使う高変動遺伝子数:", adata.n_vars)


---
## 7. PCAで情報をまとめる

single-cell RNA-seqでは、1細胞について多数の遺伝子発現量があります。

そのままでは次元が多すぎるため、まずPCAで情報をまとめます。

1-2では、PCAを使って**サンプル間の違い**を見ました。  
ここでは、同じ考え方を**細胞間の違い**に使います。


In [ ]:
sc.tl.pca(adata)

sc.pl.pca_variance_ratio(
    adata,
    log=True,
    n_pcs=30
)


---
## 8. 似た細胞同士の関係を計算する

次に、PCAの結果を使って、似ている細胞同士を近傍グラフでつなぎます。

簡単に言えば、

> 各細胞について「自分に似ている細胞は誰か」を探す

処理です。


In [ ]:
sc.pp.neighbors(
    adata,
    n_neighbors=10,
    n_pcs=40
)


---
## 9. UMAPで細胞を2次元に並べる

UMAPを使って、細胞を2次元上に配置します。

### UMAPの読み方

- 1つの点 = 1つの細胞
- 近くの点 = 発現パターンが似ている細胞
- 遠くの点 = 発現パターンが異なる細胞

ただし、UMAPの軸そのものに直接的な生物学的意味があるわけではありません。


In [ ]:
sc.tl.umap(adata)

sc.pl.umap(
    adata,
    title="UMAP of PBMC cells"
)


### ミニ演習2：クラスタ数を予想する

まだクラスタリングはしていません。

UMAPの形だけを見て、

> **いくつくらいの細胞集団がありそうか**

を予想してください。

予想したクラスタ数：

予想した理由：


---
## 10. Leiden clusteringで細胞をグループ分けする

Leiden clusteringは、似た細胞同士をグループに分ける方法です。

重要なのは、クラスタ数が最初から決まっているわけではないことです。

`resolution`という値を変えると、クラスタの細かさが変わります。


In [ ]:
# 基準となるクラスタリング
sc.tl.leiden(
    adata,
    resolution=0.5,
    key_added="leiden_r05",
    random_state=0
)

print("クラスタ数:", adata.obs["leiden_r05"].nunique())

sc.pl.umap(
    adata,
    color="leiden_r05",
    legend_loc="on data",
    title="Leiden clustering (resolution = 0.5)"
)


### ミニ演習3：resolutionを変えてみる

下の`TEST_RESOLUTION`を書き換えてください。

例えば、

- 0.2
- 0.5
- 1.0

を試します。

> resolutionを大きくすると、クラスタ数は増えるでしょうか、減るでしょうか？

先に予想してから実行してください。


In [ ]:
# ↓↓↓ ここを書き換えてください ↓↓↓
TEST_RESOLUTION = 0.8

sc.tl.leiden(
    adata,
    resolution=TEST_RESOLUTION,
    key_added="leiden_test",
    random_state=0
)

print("resolution:", TEST_RESOLUTION)
print("クラスタ数:", adata.obs["leiden_test"].nunique())

sc.pl.umap(
    adata,
    color="leiden_test",
    legend_loc="on data",
    title=f"Leiden clustering (resolution = {TEST_RESOLUTION})"
)


#### 記録してみよう

| resolution | クラスタ数 |
|---|---|
| 0.2 | |
| 0.5 | |
| 1.0 | |

- resolutionを大きくすると、どうなりましたか？
- 細かく分ければ分けるほど良い解析になると思いますか？
- 細胞型と細胞状態の違いを考えると、クラスタ数の決め方は簡単でしょうか？

このあとでは、`resolution = 0.5`の結果を使います。


---
## 11. 各クラスタのマーカー遺伝子を探す

クラスタができても、番号だけではその正体はわかりません。

そこで、

> そのクラスタで特徴的に高く発現している遺伝子

を探します。これをマーカー遺伝子と呼びます。

ここではWilcoxon rank-sum testを使って、クラスタごとの特徴的な遺伝子を調べます。


In [ ]:
sc.tl.rank_genes_groups(
    adata,
    groupby="leiden_r05",
    method="wilcoxon",
    use_raw=True
)

sc.pl.rank_genes_groups(
    adata,
    n_genes=10,
    sharey=False
)


### 見るポイント

クラスタごとに、上位に出てくる遺伝子が異なることを確認してください。

この遺伝子リストと、既知の免疫細胞マーカーを照らし合わせることで、クラスタの細胞型を推定できます。


---
## 12. 既知マーカー遺伝子をUMAP上で見る

PBMCに含まれる代表的な細胞とマーカー遺伝子の例です。

| 細胞型 | 代表的マーカー |
|---|---|
| T cell | CD3D, IL7R |
| B cell | MS4A1, CD79A |
| NK / cytotoxic cell | NKG7, GNLY |
| Monocyte | LYZ, S100A8, FCGR3A |
| Dendritic cell | FCER1A |
| Platelet | PPBP, PF4 |

まず、代表的な遺伝子をまとめてUMAP上に表示します。


In [ ]:
marker_genes = [
    "CD3D",
    "IL7R",
    "MS4A1",
    "NKG7",
    "GNLY",
    "LYZ",
    "S100A8",
    "FCGR3A",
    "FCER1A",
    "PPBP"
]

sc.pl.umap(
    adata,
    color=marker_genes,
    ncols=3
)


### ミニ演習4：自分でマーカー遺伝子を選ぶ

上の表やマーカー遺伝子ランキングを見て、気になる遺伝子を1つ選びます。

下の`GENE_TO_PLOT`を書き換えて、その遺伝子がどのクラスタで発現しているか確認してください。


In [ ]:
# ↓↓↓ 遺伝子名を書き換えてください ↓↓↓
GENE_TO_PLOT = "MS4A1"

if GENE_TO_PLOT not in adata.raw.var_names:
    raise ValueError(
        f"{GENE_TO_PLOT} はデータ中に見つかりません。"
        "遺伝子名のスペルを確認してください。"
    )

sc.pl.umap(
    adata,
    color=[GENE_TO_PLOT, "leiden_r05"],
    title=[f"Expression of {GENE_TO_PLOT}", "Leiden clusters"]
)


#### 記録してみよう

- 選んだ遺伝子：
- その遺伝子はどのクラスタで高く発現していましたか？
- そのマーカーから、どの細胞型だと推定できますか？


---
## 13. DotPlotでクラスタとマーカーの関係を見る

UMAPでは1遺伝子ずつ発現場所を確認しました。

次に、複数のマーカー遺伝子をまとめて比較します。

DotPlotでは、

- 点の大きさ：その遺伝子を発現している細胞の割合
- 点の色：平均発現量

を表します。


In [ ]:
marker_dict = {
    "T cell": ["CD3D", "IL7R"],
    "B cell": ["MS4A1", "CD79A"],
    "NK / cytotoxic": ["NKG7", "GNLY"],
    "Monocyte": ["LYZ", "S100A8", "FCGR3A"],
    "Dendritic": ["FCER1A"],
    "Platelet": ["PPBP", "PF4"]
}

sc.pl.dotplot(
    adata,
    marker_dict,
    groupby="leiden_r05",
    use_raw=True,
    standard_scale="var"
)


### 考えてみよう：クラスタの正体を推定する

DotPlotとUMAPを見て、いくつかのクラスタについて細胞型を推定してください。

| Cluster | 推定した細胞型 | 根拠となるマーカー遺伝子 |
|---|---|---|
| | | |
| | | |
| | | |
| | | |

「正解のラベルを当てる」ことだけが目的ではありません。

重要なのは、

> **どの遺伝子が、どのクラスタで高く発現しているかを根拠として説明すること**

です。


---
## 14. まとめ

このNotebookでは、single-cell RNA-seq解析の基本的な流れを体験しました。

1. single-cell RNA-seqデータを読み込む
2. QC指標を確認する
3. 質の低い細胞や情報量の少ない遺伝子を除く
4. 正規化とlog変換を行う
5. 高変動遺伝子を選ぶ
6. PCAと近傍グラフを計算する
7. UMAPで細胞を可視化する
8. Leiden clusteringで細胞をグループ分けする
9. マーカー遺伝子から細胞型を推定する

### 最後の確認

次の問いに、自分の言葉で答えてください。

**Q1. UMAP上の1つの点は何を表していますか？**

答え：

**Q2. QCはなぜ必要ですか？**

答え：

**Q3. クラスタ番号だけでは細胞型がわからないのはなぜですか？**

答え：

**Q4. 細胞型を推定するとき、何を根拠にしますか？**

答え：


---
## 発展：もっと試したい人へ

時間に余裕がある場合は、次を試してみてください。

### Challenge 1
`n_neighbors`を5、10、30に変えて、UMAPの形がどう変わるか比較する。

### Challenge 2
自分で別の免疫細胞マーカー遺伝子を調べ、UMAP上に表示する。

### Challenge 3
Leiden clusteringの`resolution`をさらに変えて、細胞集団の分かれ方を比較する。

### Challenge 4
各クラスタの細胞数を集計し、棒グラフにする。

ヒント：

```python
adata.obs["leiden_r05"].value_counts().sort_index()
```

解析パラメータを変えると結果が変わります。  
「どの設定が正解か」ではなく、**結果の違いを確認し、その理由を考えること**が大切です。
